In [1]:
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, make_jaxpr
import timeit
import numpy as np

In [2]:
jax.config.update("jax_enable_x64", False)

In [3]:
def f(x1, x2):
    return jnp.log(x1) + x1 * x2 - jnp.sin(x2)

In [4]:
# task 1
dy_dx1 = grad(f, argnums=0)  # ∂f/∂x1
dy_dx2 = grad(f, argnums=1)  # ∂f/∂x2

x1_val = 2.0
x2_val = 5.0

print("f(2.0, 5.0) =", f(x1_val, x2_val))       
print("dy_dx1(2,5) =", dy_dx1(x1_val, x2_val))   
print("dy_dx2(2,5) =", dy_dx2(x1_val, x2_val))

W1121 06:28:41.185319   11874 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W1121 06:28:41.188451   11831 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


f(2.0, 5.0) = 11.652072
dy_dx1(2,5) = 5.5
dy_dx2(2,5) = 1.7163378


In [5]:
# task 2
print("jaxpr for dy_dx1")
print(make_jaxpr(dy_dx1)(x1_val, x2_val))

print("\njaxpr for dy_dx2")
print(make_jaxpr(dy_dx2)(x1_val, x2_val))

jaxpr for dy_dx1
{ lambda ; a:f32[] b:f32[]. let
    c:f32[] = log a
    d:f32[] = mul a b
    e:f32[] = add c d
    f:f32[] = sin b
    _:f32[] = sub e f
    g:f32[] = mul 1.0:f32[] b
    h:f32[] = div 1.0:f32[] a
    i:f32[] = add_any g h
  in (i,) }

jaxpr for dy_dx2
{ lambda ; a:f32[] b:f32[]. let
    c:f32[] = log a
    d:f32[] = mul a b
    e:f32[] = add c d
    f:f32[] = sin b
    g:f32[] = cos b
    _:f32[] = sub e f
    h:f32[] = neg 1.0:f32[]
    i:f32[] = mul h g
    j:f32[] = mul a 1.0:f32[]
    k:f32[] = add_any i j
  in (k,) }


In [8]:
import numpy as np

# Inputs matching the Appendix example
x1 = 2.0
x2 = 5.0

# 1. Forward Primal Trace (Calculating f(x))
v_minus_1 = x1
v0 = x2
v3 = np.sin(v0)             # sin(5)
v2 = v_minus_1 * v0         # 2 * 5
v1 = np.log(v_minus_1)      # ln(2)
v4 = v1 + v2
v5 = v4 - v3
print(f"Forward Output (v5): {v5:.4f}")

v_bar_5 = 1.0

v_bar_4 = v_bar_5 * 1.0
v_bar_3 = v_bar_5 * (-1.0)
# print(f"v_bar_3 (Gradient of sin term): {v_bar_3}")

grad_x2_part1 = v_bar_3 * np.cos(v0)
# print(f"Intermediate x2 grad (from sin): {grad_x2_part1:.4f}")

v_bar_1 = v_bar_4 * 1.0
v_bar_2 = v_bar_4 * 1.0

grad_x1_part1 = v_bar_1 * (1.0 / v_minus_1)
grad_x1_part2 = v_bar_2 * v0
dy_dx1 = grad_x1_part1 + grad_x1_part2
print(f"Total dy/dx1: {dy_dx1:.4f}")

grad_x2_part2 = v_bar_2 * v_minus_1
# print(f"Intermediate x2 grad (from product): {grad_x2_part2:.4f}")

dy_dx2 = grad_x2_part1 + grad_x2_part2
print(f"Total dy/dx2: {dy_dx2:.4f}")

Forward Output (v5): 11.6521
Total dy/dx1: 5.5000
Total dy/dx2: 1.7163


In [10]:
dy_dx1 = grad(f, argnums=0)
dy_dx2 = grad(f, argnums=1)

print("HLO for dy_dx1")
print(jit(dy_dx1).lower(x1_val, x2_val).as_text())

print("\nHLO for dy_dx2")
print(jit(dy_dx2).lower(x1_val, x2_val).as_text())

HLO for dy_dx1
module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>, %arg1: tensor<f32>) -> (tensor<f32> {jax.result_info = "result"}) {
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f32>
    %0 = stablehlo.multiply %cst, %arg1 : tensor<f32>
    %cst_0 = stablehlo.constant dense<1.000000e+00> : tensor<f32>
    %1 = stablehlo.divide %cst_0, %arg0 : tensor<f32>
    %2 = stablehlo.add %0, %1 : tensor<f32>
    return %2 : tensor<f32>
  }
}


HLO for dy_dx2
module @jit_f attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>, %arg1: tensor<f32>) -> (tensor<f32> {jax.result_info = "result"}) {
    %0 = stablehlo.cosine %arg1 : tensor<f32>
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f32>
    %1 = stablehlo.negate %cst : tensor<f32>
    %2 = stablehlo.multiply %1, %0 : tensor<f32>
    %cst_0 = stablehlo.constant dense<1.000000

In [11]:
g1 = lambda x1, x2: (jit(f)(x1, x2),
                     jit(dy_dx1)(x1, x2),
                     jit(dy_dx2)(x1, x2))

g2 = jit(lambda x1, x2: (f(x1, x2), dy_dx1(x1, x2), dy_dx2(x1, x2)))

In [12]:
# Warmup to compile everything
print("Warming up JIT...")
g1(2.0, 5.0)
g2(2.0, 5.0)
# Block to ensure async dispatch is done
jax.block_until_ready(g2(2.0, 5.0)[0])

Warming up JIT...


Array(11.652072, dtype=float32, weak_type=True)

In [14]:
print("Timing (10,000 calls)")

# Using timeit library
t_g1 = timeit.timeit(lambda: g1(2.0, 5.0), number=10000)
t_g2 = timeit.timeit(lambda: g2(2.0, 5.0), number=10000)

print(f"g1 (separate jits) : {t_g1:.4f} s")
print(f"g2 (single jit)    : {t_g2:.4f} s")
print(f"Speedup            : {t_g1/t_g2:.2f}x")

print("\nHLO for g2 (fused)")
print(g2.lower(x1_val, x2_val).as_text())

Timing (10,000 calls)
g1 (separate jits) : 6.8874 s
g2 (single jit)    : 1.3970 s
Speedup            : 4.93x

HLO for g2 (fused)
module @jit__lambda attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<f32>, %arg1: tensor<f32>) -> (tensor<f32> {jax.result_info = "result[0]"}, tensor<f32> {jax.result_info = "result[1]"}, tensor<f32> {jax.result_info = "result[2]"}) {
    %0 = stablehlo.log %arg0 : tensor<f32>
    %1 = stablehlo.multiply %arg0, %arg1 : tensor<f32>
    %2 = stablehlo.add %0, %1 : tensor<f32>
    %3 = stablehlo.sine %arg1 : tensor<f32>
    %4 = stablehlo.subtract %2, %3 : tensor<f32>
    %cst = stablehlo.constant dense<1.000000e+00> : tensor<f32>
    %5 = stablehlo.multiply %cst, %arg1 : tensor<f32>
    %cst_0 = stablehlo.constant dense<1.000000e+00> : tensor<f32>
    %6 = stablehlo.divide %cst_0, %arg0 : tensor<f32>
    %7 = stablehlo.add %5, %6 : tensor<f32>
    %8 = stablehlo.cosine %arg1 : tensor<f32>
    %cst

In [15]:
# (a) batch over both arguments
vmap_both = vmap(g2, in_axes=(0, 0))

x1s = jnp.linspace(1.0, 10.0, 1000).astype(jnp.float32)
x2s = x1s + 1.0

result_both = vmap_both(x1s, x2s)
print("\nVmap (both) shape check:", result_both[0].shape)

# (b) batch only over x1, x2 scalar
vmap_x1_only = vmap(g2, in_axes=(0, None))

result_scalar_x2 = vmap_x1_only(x1s, 0.5)
print("Vmap (scalar x2) shape check:", result_scalar_x2[0].shape)

print("\nJaxpr: vmap_both")
print(make_jaxpr(vmap_both)(x1s, x2s))

print("\nJaxpr: vmap_x1_only")
print(make_jaxpr(vmap_x1_only)(x1s, 0.5))


Vmap (both) shape check: (1000,)
Vmap (scalar x2) shape check: (1000,)

Jaxpr: vmap_both
{ lambda ; a:f32[1000] b:f32[1000]. let
    c:f32[1000] d:f32[1000] e:f32[1000] = jit[
      name=<lambda>
      jaxpr={ lambda ; a:f32[1000] b:f32[1000]. let
          f:f32[1000] = log a
          g:f32[1000] = mul a b
          h:f32[1000] = add f g
          i:f32[1000] = sin b
          c:f32[1000] = sub h i
          j:f32[1000] = log a
          k:f32[1000] = mul a b
          l:f32[1000] = add j k
          m:f32[1000] = sin b
          _:f32[1000] = sub l m
          n:f32[1000] = mul 1.0:f32[] b
          o:f32[1000] = div 1.0:f32[] a
          d:f32[1000] = add_any n o
          p:f32[1000] = log a
          q:f32[1000] = mul a b
          r:f32[1000] = add p q
          s:f32[1000] = sin b
          t:f32[1000] = cos b
          _:f32[1000] = sub r s
          u:f32[] = neg 1.0:f32[]
          v:f32[1000] = mul u t
          w:f32[1000] = mul a 1.0:f32[]
          e:f32[1000] = add_any